# Raw SVI diagnostics

Fit one Raw SVI slice per expiry to the screened SPX midpoint-IV panel, then project total variance onto a calendar-monotone grid without refitting the slices. This notebook summarizes calibration, compares raw and repaired smiles/surfaces, and checks residuals, held-out strikes, stability, butterfly conditions, and calendar monotonicity. The checks are diagnostics; sampled grids do not prove absence of arbitrage.

Saved quote-bearing outputs are intentionally omitted from the public notebook. Run locally with licensed data; see [notebook guide](README.md).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.raw_svi import (
    fit_raw_svi_surface,
    raw_svi_butterfly_diagnostic,
    raw_svi_total_variance,
)
from src.vol_surface import (
    build_total_variance_grid,
    check_surface_quality,
    count_calendar_crossings,
    enforce_calendar_monotonicity,
    evaluate_surface,
)

PANEL_PATH = ROOT / 'data/processed/spx_iv_panel_2025-08-29.csv'
panel = pd.read_csv(PANEL_PATH, parse_dates=['quote_date', 'expiry_date'])
unconstrained_result = fit_raw_svi_surface(panel)
arbitrage_aware_result = fit_raw_svi_surface(panel, enforce_arbitrage=True)
observations = arbitrage_aware_result['observations'].copy()
parameters = arbitrage_aware_result['parameters_by_expiry'].copy()
calendar_k_grid = np.linspace(-0.5, 0.25, 151)
surface_grid = build_total_variance_grid(parameters, calendar_k_grid)
calendar_k_grid = surface_grid['log_moneyness']
surface_grid['repaired_total_variance'] = enforce_calendar_monotonicity(
    surface_grid['raw_total_variance']
)
observations['days_to_expiry'] = (observations['expiry_date'] - observations['quote_date']).dt.days
observations['fitted_iv'] = np.sqrt(observations['fitted_w'] / observations['time_to_expiry'])
observations['iv_residual'] = observations['fitted_iv'] - observations['mid_iv']
print(f'Input panel: {len(panel):,} rows; fitted OTM quotes: {len(observations):,}; expiries: {len(parameters)}')
print(f'Arbitrage-aware calibration: {arbitrage_aware_result["arbitrage_constrained"]}')

## Fit summary

Each expiry is calibrated independently by least squares in total variance, $w=\sigma^2\tau$. The table below reports the arbitrage-aware Raw SVI parameters, in-sample RMSE, and number of feasible calibration starts. The separate comparison shows the fit-cost of imposing the sampled butterfly conditions.

In [ ]:
fit_summary = parameters.merge(
    observations.groupby('expiry_date').agg(
        days_to_expiry=('days_to_expiry', 'first'),
        n_quotes=('market_w', 'size'),
    ).reset_index(),
    on='expiry_date',
)
fit_summary = fit_summary[['expiry_date', 'days_to_expiry', 'n_quotes', 'a', 'b', 'rho', 'm', 'sigma', 'rmse', 'feasible_starts', 'k_min', 'k_max']]
display(fit_summary.assign(expiry_date=fit_summary['expiry_date'].dt.strftime('%Y-%m-%d')).round(6))
display(fit_summary[['n_quotes', 'rmse']].describe().round(6))

unconstrained_rmse = unconstrained_result['parameters_by_expiry'][['expiry_date', 'rmse']].rename(columns={'rmse': 'unconstrained_rmse'})
arbitrage_rmse = parameters[['expiry_date', 'rmse']].rename(columns={'rmse': 'arbitrage_aware_rmse'})
rmse_comparison = unconstrained_rmse.merge(arbitrage_rmse, on='expiry_date', validate='one_to_one')
rmse_comparison['rmse_increase'] = rmse_comparison['arbitrage_aware_rmse'] - rmse_comparison['unconstrained_rmse']
display(rmse_comparison.assign(expiry_date=rmse_comparison['expiry_date'].dt.strftime('%Y-%m-%d')).round(7))
display(rmse_comparison[['unconstrained_rmse', 'arbitrage_aware_rmse', 'rmse_increase']].mean().to_frame('mean_rmse').round(7))

## Representative 2D smiles

Select the available expiries nearest 7, 30, 90, and 300 DTE. Points are screened market observations; each line is that expiry's fitted Raw SVI total variance curve. The line is shown only over the observed $k$ range, so this view does not disguise extrapolation as market fit.

In [ ]:
expiry_info = observations[['expiry_date', 'days_to_expiry']].drop_duplicates().sort_values('days_to_expiry')
selected_expiries = []
for target in [7, 30, 90, 300]:
    selected = expiry_info.iloc[(expiry_info['days_to_expiry'] - target).abs().argmin()]
    if selected['expiry_date'] not in [row['expiry_date'] for row in selected_expiries]:
        selected_expiries.append({'target_dte': target, **selected.to_dict()})

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
for ax, selected in zip(axes.flat, selected_expiries):
    expiry = selected['expiry_date']
    rows = observations.loc[observations['expiry_date'].eq(expiry)]
    params = parameters.loc[parameters['expiry_date'].eq(expiry)].iloc[0]
    k_grid = np.linspace(rows['log_moneyness'].min(), rows['log_moneyness'].max(), 400)
    p = [params[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    ax.scatter(rows['log_moneyness'], rows['market_w'], s=13, color='black', alpha=0.65, label='Observed')
    ax.plot(k_grid, raw_svi_total_variance(k_grid, *p), color='tab:blue', lw=2, label='Raw SVI fit')
    expiry_index = np.flatnonzero(np.isclose(surface_grid['time_to_expiry'], params['time_to_expiry']))[0]
    ax.plot(calendar_k_grid, surface_grid['repaired_total_variance'][expiry_index], color='tab:orange', lw=2, label='Calendar repaired')
    ax.set(title=f'{expiry.date()} ({selected["days_to_expiry"]} DTE; target {selected["target_dte"]})', xlabel='Log-moneyness k', ylabel='Total variance w')
    ax.grid(alpha=0.25)
    ax.legend()
for ax in axes.flat[len(selected_expiries):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## Observed and fitted 3D total-variance surface

Rotate the figure to inspect observed quotes, the fitted Raw SVI slices, and their calendar-repaired total-variance grid. Both meshes are masked to each expiry's observed $k$ range; unsupported extrapolated wings are not shown as repaired surface.

In [ ]:
k_mesh_grid = np.linspace(observations['log_moneyness'].min(), observations['log_moneyness'].max(), 180)
expiry_days = fit_summary['days_to_expiry'].to_numpy()
w_mesh = np.full((len(fit_summary), len(k_mesh_grid)), np.nan)
for i, params in fit_summary.iterrows():
    inside = (k_mesh_grid >= params['k_min']) & (k_mesh_grid <= params['k_max'])
    p = [params[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    w_mesh[i, inside] = raw_svi_total_variance(k_mesh_grid[inside], *p)

fig = go.Figure()
fig.add_trace(go.Surface(
    x=k_mesh_grid, y=expiry_days, z=w_mesh, colorscale='Viridis', opacity=0.68,
    connectgaps=False, name='Fitted Raw SVI', colorbar=dict(title='Fitted w'),
))
fig.add_trace(go.Surface(
    x=surface_grid['log_moneyness'],
    y=surface_grid['time_to_expiry'] * 365,
    z=surface_grid['repaired_total_variance'],
    colorscale='Cividis', opacity=0.48, showscale=False, name='Calendar repaired',
))
fig.add_trace(go.Scatter3d(
    x=observations['log_moneyness'], y=observations['days_to_expiry'], z=observations['market_w'],
    mode='markers', marker=dict(size=2, color='black', opacity=0.6), name='Observed w',
    customdata=np.column_stack([observations['fitted_w'], observations['residual_w']]),
    hovertemplate='k=%{x:.3f}<br>DTE=%{y:.0f}<br>Observed w=%{z:.5f}<br>Fitted w=%{customdata[0]:.5f}<br>Residual=%{customdata[1]:+.5f}<extra></extra>',
))
fig.update_layout(title='Observed quotes and fitted Raw SVI slices', width=1000, height=750, scene=dict(xaxis_title='Log-moneyness k', yaxis_title='Days to expiry', zaxis_title='Total variance w'))
fig.show()

## Residual diagnostics

Residuals are fitted minus observed total variance. Look for systematic patterns with moneyness, maturity, or fitted level; a low aggregate RMSE can conceal structured errors. The second plot expresses errors in IV points for easier market interpretation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(observations['log_moneyness'], observations['residual_w'], s=8, alpha=0.35, color='black')
axes[0].axhline(0, color='tab:red', ls='--')
axes[0].set(xlabel='Log-moneyness k', ylabel='Residual in w', title='Residual vs moneyness')
axes[1].scatter(observations['days_to_expiry'], observations['residual_w'], s=8, alpha=0.35, color='black')
axes[1].axhline(0, color='tab:red', ls='--')
axes[1].set(xlabel='Days to expiry', ylabel='Residual in w', title='Residual vs maturity')
axes[2].scatter(observations['mid_iv'], observations['iv_residual'], s=8, alpha=0.35, color='black')
axes[2].axhline(0, color='tab:red', ls='--')
axes[2].set(xlabel='Observed midpoint IV', ylabel='Residual in IV', title='IV residual vs observed IV')
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()
display(observations[['residual_w', 'iv_residual']].describe().round(6))

## Held-out-strike validation

Within each expiry, sort by $k$ and hold out every fifth quote. Fit on the remaining quotes and evaluate the withheld strikes. This tests interpolation across the observed smile; it does not test maturity extrapolation. Compare train and test RMSE per expiry to spot overfitting.

In [ ]:
holdout_rows = []
for expiry, rows in observations.groupby('expiry_date', sort=True):
    rows = rows.sort_values('log_moneyness').reset_index(drop=True)
    test_mask = np.arange(len(rows)) % 5 == 0
    train, test = rows.loc[~test_mask].copy(), rows.loc[test_mask].copy()
    fitted = fit_raw_svi_surface(train, enforce_arbitrage=True)['parameters_by_expiry'].iloc[0]
    p = [fitted[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    test_w = raw_svi_total_variance(test['log_moneyness'].to_numpy(), *p)
    holdout_rows.append({
        'expiry_date': expiry, 'n_train': len(train), 'n_test': len(test),
        'train_rmse_w': fitted['rmse'],
        'test_rmse_w': float(np.sqrt(np.mean((test_w - test['market_w'].to_numpy()) ** 2))),
    })
holdout = pd.DataFrame(holdout_rows)
display(holdout.assign(expiry_date=holdout['expiry_date'].dt.strftime('%Y-%m-%d')).round(7))
display(holdout[['train_rmse_w', 'test_rmse_w']].describe().round(7))

## Calibration stability

Refit representative expiries from deliberately different starting parameters. Compare both parameter estimates and the fitted curve over the observed strike range. Parameters can vary while the fitted curves remain close, indicating weak parameter identification rather than materially different fits.

In [ ]:
stability_rows = []
for selected in selected_expiries:
    expiry = selected['expiry_date']
    rows = observations.loc[observations['expiry_date'].eq(expiry)]
    k = rows['log_moneyness'].to_numpy()
    grid = np.linspace(k.min(), k.max(), 300)
    min_w = float(rows['market_w'].min())
    starts = {
        'default': None,
        'negative_skew': [max(min_w - 0.01, -1.0), 0.03, -0.8, 0.0, 0.05],
        'symmetric': [max(min_w - 0.01, -1.0), 0.15, 0.0, 0.0, 0.2],
        'positive_skew': [max(min_w - 0.01, -1.0), 0.3, 0.7, 0.0, 0.35],
    }
    reference = None
    for name, start in starts.items():
        try:
            params = fit_raw_svi_surface(rows, initial_guess=start, enforce_arbitrage=True)['parameters_by_expiry'].iloc[0]
            p = [params[col] for col in ['a', 'b', 'rho', 'm', 'sigma']]
            curve = raw_svi_total_variance(grid, *p)
            if reference is None: reference = curve
            stability_rows.append({
                'expiry_date': expiry, 'start': name, 'rmse_w': params['rmse'],
                'max_abs_curve_difference': float(np.max(np.abs(curve - reference))),
                **dict(zip(['a', 'b', 'rho', 'm', 'sigma'], p)),
            })
        except ValueError as error:
            stability_rows.append({'expiry_date': expiry, 'start': name, 'error': str(error)})
stability = pd.DataFrame(stability_rows)
display(stability.assign(expiry_date=stability['expiry_date'].dt.strftime('%Y-%m-%d')).round(6))

## Butterfly diagnostics

For each expiry, evaluate total-variance positivity, Durrleman's $g(k)$ on the quoted interval and on the wider $[-5,5]$ grid, plus both wing constraints $b(1+\rho)<2$ and $b(1-\rho)<2$. A violation only on the wide grid is an extrapolation warning; a violation within quoted strikes is more directly relevant. Grid checks are not a formal proof over all real $k$.

In [ ]:
butterfly_rows = []
wide_grid = np.linspace(-5.0, 5.0, 4001)
for _, params in parameters.iterrows():
    rows = observations.loc[observations['expiry_date'].eq(params['expiry_date'])]
    p = [params[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
    quoted_grid = np.linspace(rows['log_moneyness'].min(), rows['log_moneyness'].max(), 1001)
    quoted = raw_svi_butterfly_diagnostic(*p, quoted_grid)
    wide = raw_svi_butterfly_diagnostic(*p, wide_grid)
    butterfly_rows.append({
        'expiry_date': params['expiry_date'], 'k_min': quoted_grid.min(), 'k_max': quoted_grid.max(),
        'min_g_quoted': quoted['minimum_g_on_grid'], 'min_g_wide': wide['minimum_g_on_grid'],
        'g_nonnegative_quoted': quoted['g_nonnegative_on_grid'],
        'g_nonnegative_wide': wide['g_nonnegative_on_grid'],
        'variance_nonnegative': wide['nonnegative_total_variance'],
        'left_wing_slope': wide['left_wing_slope'], 'left_wing_pass': wide['left_wing_condition'],
        'right_wing_slope': wide['right_wing_slope'], 'right_wing_pass': wide['right_wing_condition'],
    })
butterfly = pd.DataFrame(butterfly_rows)
display(butterfly.assign(expiry_date=butterfly['expiry_date'].dt.strftime('%Y-%m-%d')).round(7))
display(butterfly[['g_nonnegative_quoted', 'g_nonnegative_wide', 'variance_nonnegative', 'left_wing_pass', 'right_wing_pass']].mean())

## Calendar diagnostics

At each $k$, project only the maturities whose observed quote ranges include that $k$ onto the nearest nondecreasing sequence using equal-weight isotonic regression (PAVA). Unsupported SVI values are left out of the grid, so cannot influence the repair. Raw SVI slices are not refit. Each target maturity uses its fixed surrounding expiries and only their shared sampled support; an exact fitted expiry uses its own support. The evaluator never switches to more distant expiries to extend a smile.

Calendar counts compare consecutive supported maturities at each $k$, including across missing slices. Butterfly checks price calls using the actual evaluator, then test decreasing prices and convexity in strike on a dense grid including points around interpolation knots. We check fitted expiries and 30D/60D/90D targets. Linear interpolation of total variance can violate butterfly conditions between grid points; positive finite-difference $g(k)$ at the original nodes did not establish that the evaluated surface was safe. These checks expose that limitation without changing the fitted or repaired values.

The date/tenor quality table always includes 30D, 60D and 90D, with `use_for_skew` and a failure reason. Unsupported or failing tenors are flagged rather than silently omitted. `interpolated_smiles` retains evaluable failures for inspection; `usable_target_smiles` excludes them from subsequent feature extraction. No skew formula is defined yet. `skew_k_range=None` checks the full shared support; set its bounds once the feature's moneyness range is chosen. Across future quote dates, retain these quality tables and inspect failure frequency and concentration in volatile periods before excluding observations. Passing means passing sampled checks, not guaranteed arbitrage freedom.

In [ ]:
raw_w = surface_grid['raw_total_variance']
repaired_w = surface_grid['repaired_total_variance']
adjustment_w = repaired_w - raw_w
maturities = surface_grid['time_to_expiry']
raw_iv = np.sqrt(raw_w / maturities[:, None])
repaired_iv = np.sqrt(repaired_w / maturities[:, None])
adjustment_iv_points = 100 * (repaired_iv - raw_iv)
ordered_fit_summary = fit_summary.sort_values('days_to_expiry').reset_index(drop=True)
quote_coverage = surface_grid['support_mask']
quoted_adjustment_w = np.where(quote_coverage, adjustment_w, np.nan)
quoted_adjustment_iv = np.where(quote_coverage, adjustment_iv_points, np.nan)
raw_calendar_difference = np.diff(raw_w, axis=0)
repaired_calendar_difference = np.diff(repaired_w, axis=0)

calendar_rows = []
for i in range(len(maturities) - 1):
    near = ordered_fit_summary.iloc[i]
    far = ordered_fit_summary.iloc[i + 1]
    overlap = surface_grid['support_mask'][i] & surface_grid['support_mask'][i + 1]
    raw_overlap = raw_calendar_difference[i, overlap]
    repaired_overlap = repaired_calendar_difference[i, overlap]
    calendar_rows.append({
        'near_expiry': near['expiry_date'], 'near_dte': near['days_to_expiry'],
        'far_expiry': far['expiry_date'], 'far_dte': far['days_to_expiry'],
        'raw_crossings_in_quote_overlap': int((raw_overlap < -1e-10).sum()),
        'repaired_crossings_in_quote_overlap': int((repaired_overlap < -1e-10).sum()),
        'repair_changed_pair': bool(np.any(np.abs(adjustment_w[i:i + 2, overlap]) > 1e-12)) if overlap.any() else False,
    })
calendar = pd.DataFrame(calendar_rows)
display(calendar.assign(near_expiry=calendar['near_expiry'].dt.strftime('%Y-%m-%d'), far_expiry=calendar['far_expiry'].dt.strftime('%Y-%m-%d')))

adjustment_summary = pd.Series({
    'max_abs_adjustment_w_on_support': np.nanmax(np.abs(adjustment_w)),
    'median_abs_adjustment_w_on_support': np.nanmedian(np.abs(adjustment_w)),
    'median_nonzero_abs_adjustment_w': np.median(np.abs(adjustment_w)[np.abs(adjustment_w) > 1e-12]),
    'max_abs_adjustment_iv_points_on_support': np.nanmax(np.abs(adjustment_iv_points)),
    'median_abs_adjustment_iv_points_on_support': np.nanmedian(np.abs(adjustment_iv_points)),
    'raw_crossings_in_quote_overlap': calendar['raw_crossings_in_quote_overlap'].sum(),
    'repaired_crossings_in_quote_overlap': calendar['repaired_crossings_in_quote_overlap'].sum(),
    'raw_crossings_on_supported_grid': count_calendar_crossings(raw_w),
    'repaired_crossings_on_supported_grid': count_calendar_crossings(repaired_w),
    'adjacent_pairs_changed_in_quote_overlap': calendar['repair_changed_pair'].sum(),
})
display(adjustment_summary.to_frame('value').round(8))
maturity_adjustments = []
for i, params in parameters.sort_values('time_to_expiry').reset_index(drop=True).iterrows():
    max_iv_column = int(np.nanargmax(np.abs(adjustment_iv_points[i]))) if quote_coverage[i].any() else None
    maturity_adjustments.append({
        'expiry_date': params['expiry_date'],
        'max_abs_adjustment_w_on_support': np.nanmax(np.abs(adjustment_w[i])) if quote_coverage[i].any() else np.nan,
        'median_abs_adjustment_w_on_support': np.nanmedian(np.abs(adjustment_w[i])) if quote_coverage[i].any() else np.nan,
        'max_abs_adjustment_iv_points_on_support': np.nanmax(np.abs(adjustment_iv_points[i])) if quote_coverage[i].any() else np.nan,
        'k_at_max_iv_adjustment': calendar_k_grid[max_iv_column] if max_iv_column is not None else np.nan,
    })
maturity_adjustments = pd.DataFrame(maturity_adjustments)
display(maturity_adjustments.assign(expiry_date=maturity_adjustments['expiry_date'].dt.strftime('%Y-%m-%d')).round(7))
display(calendar.loc[calendar['repair_changed_pair'], ['near_expiry', 'far_expiry', 'raw_crossings_in_quote_overlap', 'repair_changed_pair']])

evaluated_butterfly_rows = [
    check_surface_quality(tau, surface_grid)
    for tau in np.unique(np.r_[maturities, np.array([30, 60, 90]) / 365])
]
evaluated_butterfly = pd.DataFrame(evaluated_butterfly_rows)
evaluated_butterfly.insert(0, 'days_to_expiry', evaluated_butterfly['time_to_expiry'] * 365)
display(evaluated_butterfly.round(8))
print('Evaluated smiles with sampled convexity violations:', int(evaluated_butterfly['convexity_violations'].gt(0).sum()))

skew_k_range = None
target_quality = pd.DataFrame([
    {
        'quote_date': panel['quote_date'].iloc[0],
        'target_dte': target_dte,
        **check_surface_quality(target_dte / 365, surface_grid, k_range=skew_k_range),
    }
    for target_dte in [30, 60, 90]
])
display(target_quality[['quote_date', 'target_dte', 'use_for_skew', 'failure_reason', 'k_min', 'k_max']])
print('Target tenors flagged for inspection:', int((~target_quality['use_for_skew']).sum()))

target_smiles = []
for quality in target_quality.to_dict('records'):
    if quality['n_test_strikes'] == 0:
        continue
    inside = (calendar_k_grid >= quality['k_min']) & (calendar_k_grid <= quality['k_max'])
    target_k = np.unique(np.r_[quality['k_min'], calendar_k_grid[inside], quality['k_max']])
    evaluated = evaluate_surface(target_k, quality['time_to_expiry'], surface_grid)
    target_smiles.append(pd.DataFrame({
        'quote_date': quality['quote_date'],
        'target_dte': quality['target_dte'],
        'log_moneyness': target_k,
        'total_variance': evaluated['total_variance'],
        'implied_volatility': evaluated['implied_volatility'],
        'use_for_skew': quality['use_for_skew'],
        'failure_reason': quality['failure_reason'],
    }))
interpolated_smiles = pd.concat(target_smiles, ignore_index=True) if target_smiles else pd.DataFrame(
    columns=['quote_date', 'target_dte', 'log_moneyness', 'total_variance', 'implied_volatility', 'use_for_skew', 'failure_reason']
)
usable_target_smiles = interpolated_smiles.loc[interpolated_smiles['use_for_skew'].eq(True)].copy()
display(usable_target_smiles.groupby('target_dte')['implied_volatility'].agg(['min', 'median', 'max']).round(6))
